# WACV 2027 Workshop on Real-World Surveillance (RWS) — Track 2: ABPR
## EXP-006: Standalone End-to-End Metric Learning & Attribute Recognition (Swin-T + ArcFace + InfoNCE)

> **Target Hardware:** Kaggle GPU T4 x 2 (32GB VRAM Total, 30GB CPU RAM)
> **Objective:** True end-to-end multi-task training on images for 25 full epochs. Jointly trains Swin-Transformer Tiny backbone + 40-attr PAR classifier + 512-D ArcFace head over 7,204 identities + 512-D InfoNCE query head.
> **Key Features:** ZERO reliance on previous experiment submission zips. 100% self-contained. Vectorized 2D memory matching prevents 18.5GB RAM leaks.
> **Execution Mode:** Background Run via **Save Version -> Save & Run All (Commit)** or Interactive Execution.


In [ ]:
# ==============================================================================
# 1. MASTER HYPERPARAMETER CONFIGURATION (Easily Editable)
# ==============================================================================
import os

CONFIG = {
    "EXPERIMENT_ID": "EXP-006",
    "MODEL_BACKBONE": "swin_t",                  # Swin-Transformer Tiny Backbone
    "EPOCHS": 25,                               # 25 full end-to-end epochs on images
    "BATCH_SIZE": 128,                          # Dual T4 batch size (64 per GPU)
    "LR_BACKBONE": 1e-4,                        # Backbone fine-tuning learning rate
    "LR_HEADS": 1e-3,                           # Classifier, ArcFace & InfoNCE heads LR
    "WEIGHT_DECAY": 1e-4,                       # AdamW weight decay
    "ARCFACE_SCALE": 30.0,                      # ArcFace hypersphere radius (s)
    "ARCFACE_MARGIN": 0.20,                     # Additive angular margin (m)
    "LAMBDA_ARC": 0.5,                          # ArcFace loss weight
    "LAMBDA_CM": 0.25,                          # InfoNCE query-image contrastive loss weight
    "LAMBDA_METRIC": 0.25,                      # Hybrid score fusion: (1-λ)*L1 + λ*cos_dist
    "NUM_WORKERS": 4,                           # High-throughput DataLoader workers
    "DATA_DIR": "/kaggle/input/datasets/hinewk/upar-challenge-2027-data", # Base path
    "CHECKPOINT_DIR": "checkpoints",
    "OUTPUT_ZIP": "submissions/EXP-006_submission.zip",
}

print(f"Master Configuration loaded for {CONFIG['EXPERIMENT_ID']} ({CONFIG['MODEL_BACKBONE']})")


In [ ]:
# ==============================================================================
# 2. PRIVATE GITHUB REPOSITORY INGESTION (Auto-Clone / Pull with Token Support)
# ==============================================================================
import os
import sys

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = "YOUR_GITHUB_PAT_TOKEN_HERE"

if GITHUB_TOKEN and GITHUB_TOKEN != "YOUR_GITHUB_PAT_TOKEN_HERE":
    REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Aditya-Lingam-9000/upar_challenge_26_rws27.git"
else:
    REPO_URL = "https://github.com/Aditya-Lingam-9000/upar_challenge_26_rws27.git"

REPO_DIR = "/kaggle/working/upar_challenge_26_rws27"

if os.path.exists(REPO_DIR):
    print("Repository directory exists. Pulling latest commits from main...")
    os.system(f"cd {REPO_DIR} && git pull origin main")
else:
    print("Cloning repository from GitHub...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"Active Working Directory: {os.getcwd()}")
!pip install -q -r requirements.txt


In [ ]:
# ==============================================================================
# 3. HARDWARE & DATASET AUTO-DETECTION
# ==============================================================================
import glob
import os
import torch
from src.dataset import resolve_data_root

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
num_gpus = torch.cuda.device_count()
print(f"Detected GPUs: {num_gpus}")
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | Total Memory: {props.total_memory / (1024**3):.2f} GB")

# Auto-resolve dataset directory
CONFIG["DATA_DIR"] = resolve_data_root(CONFIG["DATA_DIR"])
print(f"[DATASET] Final Resolved Path: {CONFIG['DATA_DIR']}")
train_csv_path = os.path.join(CONFIG["DATA_DIR"], "annotations", "train.csv")
print(f"[DATASET] train.csv verified: {os.path.exists(train_csv_path)}")


In [ ]:
# ==============================================================================
# 4. LAUNCH STANDALONE END-TO-END METRIC LEARNING PIPELINE (25 Full Epochs)
# ==============================================================================
# Jointly trains Swin-T backbone + 40-attr classifier + 512-D ArcFace head + InfoNCE query head
# ZERO external zip dependencies. Automatically packages submissions/EXP-006_submission.zip
!python train_stage2_end2end.py \
    --data_root {CONFIG["DATA_DIR"]} \
    --epochs {CONFIG["EPOCHS"]} \
    --batch_size {CONFIG["BATCH_SIZE"]} \
    --lr_backbone {CONFIG["LR_BACKBONE"]} \
    --lr_heads {CONFIG["LR_HEADS"]} \
    --weight_decay {CONFIG["WEIGHT_DECAY"]} \
    --scale {CONFIG["ARCFACE_SCALE"]} \
    --margin {CONFIG["ARCFACE_MARGIN"]} \
    --lambda_arc {CONFIG["LAMBDA_ARC"]} \
    --lambda_cm {CONFIG["LAMBDA_CM"]} \
    --lambda_metric {CONFIG["LAMBDA_METRIC"]} \
    --num_workers {CONFIG["NUM_WORKERS"]} \
    --save_dir {CONFIG["CHECKPOINT_DIR"]} \
    --output_zip {CONFIG["OUTPUT_ZIP"]}


In [ ]:
# ==============================================================================
# 5. EXECUTE CODABENCH CONTAINER SANDBOX SIMULATION (Zero Failure Guarantee)
# ==============================================================================
# Simulates exact Codabench Ingestion and Scoring program inside isolated sandbox
!python tests/test_codabench_simulation.py --zip_path {CONFIG["OUTPUT_ZIP"]} --data_root {CONFIG["DATA_DIR"]}


In [ ]:
# ==============================================================================
# 6. EXPORT SUBMISSION ARCHIVE & ONE-CLICK DOWNLOAD LINK
# ==============================================================================
import shutil
from IPython.display import FileLink, display

os.makedirs("/kaggle/working/final_submissions", exist_ok=True)
sub_dst = "/kaggle/working/final_submissions/EXP-006_submission.zip"

if os.path.exists(CONFIG["OUTPUT_ZIP"]):
    shutil.copy2(CONFIG["OUTPUT_ZIP"], sub_dst)
    zip_size_mb = os.path.getsize(sub_dst) / (1024 * 1024)
    print(f">> EXP-006 Submission Package Ready: {sub_dst} ({zip_size_mb:.1f} MB)")
    display(FileLink("final_submissions/EXP-006_submission.zip"))
    print("\nYou can now click the download link above or download from the Kaggle Output tab to submit to Codabench!")
else:
    print(f"[ERROR] Submission zip not found at {CONFIG['OUTPUT_ZIP']}")
